# 樸素貝氏（Naive Bayes）

- **核心原理**：貝氏定理＋特徵獨立假設
- **變體**：Multinomial NB（詞的「次數」）、Bernoulli NB（詞「有沒有出現」）、Gaussian NB（連續特徵）；延伸：Complement NB
- **推薦資料集**：SMS Spam Collection（5,574 則英文簡訊；目標：是不是垃圾簡訊 spam）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

跟前五章最大的不同：前面的模型都是**判別式**（discriminative）—— 直接學「看到 $x$，該猜哪一類」（邏輯回歸學 $P(y \mid x)$、KNN 找鄰居、決策樹切空間）。
樸素貝氏是**生成式**（generative）：先學「每一類的資料長什麼樣子」$P(x \mid y)$ 和「每一類有多常見」$P(y)$，預測時再用貝氏定理反推 $P(y \mid x)$。

而且「學」只是**數數**：數每一類有幾則訊息、每個詞在每一類出現幾次。沒有梯度、沒有迭代、沒有搜尋 —— 這一章的重點是：**怎麼把機率公式翻成數數、平滑為什麼不能少、為什麼一定要在 log 空間算，以及「樸素」的獨立假設錯了會怎樣。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、3D、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一筆是一則簡訊的**原始文字**（字串），標籤 ham（正常）= 0、spam（垃圾）= 1。文字不能直接丟進模型，第 2 節用 `CountVectorizer` 把每則簡訊變成「每個詞出現幾次」的向量（詞袋，bag of words）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$ | 訓練樣本（簡訊）數 | 約 4,459（以載入結果為準） |
| $V$ | 詞彙表大小（不同的詞的個數）= 特徵數 | 約 7,700 |
| $K$ | 類別數；$k$ 是類別編號（0 = ham、1 = spam） | 2 |
| $x_{ij}$ | 第 $i$ 則簡訊裡，第 $j$ 個詞出現的次數 | 大多是 0 |
| $X$ | 詞頻矩陣，第 $i$ 列是第 $i$ 則簡訊 | $(n, V)$，稀疏 |
| $\pi_k$ | 先驗（prior）：第 $k$ 類的比例 $P(y = k)$ | — |
| $\theta_{kj}$ | 第 $k$ 類的簡訊裡，「隨便抽一個詞，抽到第 $j$ 個詞」的機率（Multinomial） | $(K, V)$ |
| $p_{kj}$ | 第 $k$ 類的簡訊裡，第 $j$ 個詞「有出現」的機率（Bernoulli） | $(K, V)$ |
| $\alpha$ | 平滑（smoothing）參數；$\alpha = 1$ 叫 Laplace 平滑 | — |
| JLL | joint log-likelihood：$\log P(y = k) + \log P(x \mid y = k)$，每一筆、每一類一個數 | $(n, K)$ |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第三章 | 第一部分第 3 節：邏輯回歸（`01-classic-ml/03-logistic-regression`） | 直接學 $P(y = 1 \mid x) = \sigma(w \cdot x + b)$；用梯度下降最小化 log loss。 |
| 第三章「數值穩定的 softmax」 | 同上，TODO 3、11 | softmax 對每一列加減同一個常數結果不變 → 先減每列最大值，$e^z$ 就不會 overflow。 |
| 第三章「log loss baseline = 熵」 | 同上，TODO 2 | 只輸出類別比例的 log loss = 標籤分布的熵 $-\sum_k q_k \ln q_k$。 |
| 第三章的不平衡資料 | 同上 | 類別不平衡時 accuracy 會誤導，要看少數類的 precision／recall。 |
| 第四章 | 第一部分第 4 節：KNN | 維度詛咒：特徵很多、而且大多無關時，距離失去意義。 |
| 第四章、第五章的 1-SE | KNN、決策樹 | 跟 CV 最佳差不到一個標準差的候選中挑最簡單的。 |
| 第五章 | 第一部分第 5 節：決策樹（`01-classic-ml/05-decision-tree`） | 熵 $H = -\sum_k p_k \log_2 p_k$；葉子的機率是「數出來的比例」；完全長滿時機率只有 0／1，log loss 很差。 |
| 第五章的不穩定性 | 同上，TODO 15 | 高變異：換一批訓練資料預測就大變；平均很多棵樹可以降低變異。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 條件機率**：$P(A \mid B) = \dfrac{P(A \cap B)}{P(B)}$ = 「已知 $B$ 發生，$A$ 也發生的比例」。例：100 則簡訊、13 則 spam；含 "free" 的有 10 則，其中 8 則是 spam → $P(\text{spam} \mid \text{free}) = 8/10 = 0.8$，$P(\text{free} \mid \text{spam}) = 8/13 \approx 0.62$。**兩者不一樣**。

**2. 貝氏定理**：
$$P(y \mid x) = \frac{P(x \mid y)\,P(y)}{P(x)}, \qquad P(x) = \sum_k P(x \mid y = k)\,P(y = k)$$
例（接上）：$P(\text{free} \mid \text{spam}) P(\text{spam}) = \frac{8}{13} \cdot \frac{13}{100} = 0.08$；$P(\text{free} \mid \text{ham}) P(\text{ham}) = \frac{2}{87} \cdot \frac{87}{100} = 0.02$ → $P(\text{spam} \mid \text{free}) = \frac{0.08}{0.08 + 0.02} = 0.8$ ✓。
分母 $P(x)$ 對每一類都一樣，**比大小時可以不算**；要機率時再除以「所有類別的分子總和」。

**3. 獨立**：$A$、$B$ 獨立 ⇔ $P(A \cap B) = P(A)\,P(B)$。**條件獨立**：已知 $y$ 之後，$P(x_1, x_2 \mid y) = P(x_1 \mid y)\,P(x_2 \mid y)$。例：擲兩次公平硬幣，兩次都正面 $= 0.5 \times 0.5$。

**4. 對數的規則**：$\log(ab) = \log a + \log b$、$\log(a^c) = c \log a$、$\log(a / b) = \log a - \log b$；$\log$ 是單調遞增，比大小時「比原數」和「比 log」結論一樣。

**5. 浮點數的下限**：float64 最小的正常數約 $2.2 \times 10^{-308}$（`np.finfo(float).tiny`）；更小就變成 0（underflow）。$0.01^{200} = 10^{-400}$ 會直接變成 0。

**6. log-sum-exp**：要算 $\log \sum_k e^{z_k}$ 時，先減掉最大值 $m = \max_k z_k$：
$$\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$$
括號裡最大的一項是 $e^0 = 1$，不會 overflow 也不會整個 underflow。例：$z = (-1000, -1001)$ → 直接算 $e^{-1000} = 0$、$\log 0 = -\infty$ ✗；減最大值 → $-1000 + \log(1 + e^{-1}) = -999.687$ ✓。（跟第三章「softmax 先減最大值」是同一招。）

**7. 用比例估計機率（最大概似估計 MLE）**：一枚硬幣擲 10 次、正面 7 次，「正面機率」的 MLE 是 $7/10$。類別分布也一樣：袋子裡各種詞的機率 = 各自出現的次數 ÷ 總次數。

**8. 常態分布的密度**：$\mathcal{N}(x; \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$，取 log：$-\frac{1}{2}\log(2\pi\sigma^2) - \frac{(x - \mu)^2}{2\sigma^2}$。例：$x = \mu$ 時第二項是 0。

**9. 偏差—變異**（重述）：偏差高 = 模型假設太強、連訓練資料都學不好；變異高 = 對訓練資料太敏感。**假設越強（越「樸素」）→ 偏差高、變異低**。

#### 程式

**10. 稀疏矩陣（`scipy.sparse.csr_matrix`）**：只存非零的位置與值。7,700 個詞、每則簡訊只用到十幾個 → 99.8% 是 0。

```python
from scipy.sparse import csr_matrix
A = csr_matrix(np.array([[0, 2, 0], [1, 0, 0]]))
A.nnz            # 2：非零元素個數
A @ np.ones(3)   # 一般的矩陣乘法照用 → array([2., 1.])
A.toarray()      # 轉回一般的 ndarray（很大時會吃光記憶體，小心）
A.sum(axis=1)    # 回傳 np.matrix (2, 1)，用 np.asarray(...).ravel() 轉成 (2,)
```

**11. one-hot 與矩陣乘法做「分組加總」**：`Y = np.eye(K)[y]` 是 $(n, K)$。`Y.T @ X` 的第 $(k, j)$ 格 $= \sum_i Y_{ik} X_{ij}$ = 「第 $k$ 類的所有列」在第 $j$ 欄的總和。

```python
y = np.array([0, 1, 0]); X = np.array([[1, 2], [3, 4], [5, 6]])
np.eye(2)[y].T @ X    # [[1+5, 2+6], [3, 4]] = [[6, 8], [3, 4]]
```

**12. `np.log1p(x)` = $\log(1 + x)$**，$x$ 很小時比 `np.log(1 + x)` 準；`np.log1p(-p)` = $\log(1 - p)$。

**13. `scipy.special.logsumexp(Z, axis=1, keepdims=True)`**：每一列做第 6 點的計算。

**14. pandas 字串方法**：`s.str.len()` 長度、`s.str.count(r"\d")` 數字的個數（regex）、`s.str.contains("free", case=False)` 是否包含（不分大小寫）。

**15. `CountVectorizer`**：`fit` 建詞彙表（預設：轉小寫、只保留 2 個字元以上的英數字詞）；`transform` 把每則文字變成一列詞頻；`vocabulary_` 是 {詞: 欄位編號}、`get_feature_names_out()` 是欄位編號 → 詞。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。每格都附了「看什麼」的提示，請把看到的數字與結論寫進去。

第一格先把載入結果整理成固定格式：`X_*` 是文字的 `pd.Series`，`y_*` 是 0（ham）／1（spam）的整數。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_sms_spam

X_train, X_test, y_train, y_test = load_sms_spam()


def as_text(X) -> pd.Series:
    """載入結果 → 文字的 pd.Series（DataFrame 時取第一個文字欄）。"""
    if isinstance(X, pd.DataFrame):
        X = X[X.select_dtypes(exclude="number").columns[0]]
    return pd.Series(X).astype(str)


def as_label(y) -> pd.Series:
    """載入結果 → 0／1 的 pd.Series（字串標籤時 spam = 1）。"""
    y = pd.Series(y)
    if not pd.api.types.is_numeric_dtype(y):
        y = (y.astype(str).str.lower() == "spam").astype(int)
    return y.astype(int)


X_train, X_test = as_text(X_train), as_text(X_test)
y_train, y_test = as_label(y_train), as_label(y_test)
train = pd.DataFrame({"text": X_train, "spam": y_train})

print("train / test：", X_train.shape, X_test.shape)
pd.set_option("display.max_colwidth", 120)
display(train.groupby("spam").head(3).sort_values("spam"))

In [ ]:
# 類別分布與重複
# 觀察：（spam 佔多少？「全部猜 ham」的 accuracy 是多少？這個數字為什麼不能代表模型好？
#        有多少則簡訊一字不差地重複出現？重複最多的是 ham 還是 spam？）
display(
    pd.DataFrame(
        {"train": y_train.value_counts(normalize=True).sort_index(), "test": y_test.value_counts(normalize=True).sort_index()}
    ).rename(index={0: "ham", 1: "spam"}).round(3)
)
dup = X_train.duplicated(keep=False)
print(f"train 裡內容重複的簡訊：{dup.sum()} 則（{dup.mean():.1%}）")
display(train[dup].groupby(["text", "spam"]).size().sort_values(ascending=False).head(5).rename("count").reset_index())
print("同時出現在 train 與 test 的文字：", len(set(X_train) & set(X_test)), "種")

In [ ]:
# 簡單的數值特徵：長度、數字、大寫、驚嘆號、網址（3C 的 Gaussian NB 會用到）
# 觀察：（哪幾個特徵 spam 和 ham 差最多？分布長什麼樣子 —— 對稱的鐘形，還是右偏、一大堆 0？
#        這對「每一類的每個特徵都是常態分布」的假設有什麼影響？）
def text_features(s: pd.Series) -> pd.DataFrame:
    """文字 → 6 個數值特徵 (n, 6)。"""
    return pd.DataFrame(
        {
            "length": s.str.len(),
            "n_words": s.str.split().str.len(),
            "n_digits": s.str.count(r"\d"),
            "n_upper": s.str.count(r"[A-Z]"),
            "n_excl": s.str.count("!"),
            "has_url": s.str.contains(r"http|www", case=False).astype(int),
        },
        index=s.index,
    ).astype(float)


F_train, F_test = text_features(X_train), text_features(X_test)
display(F_train.groupby(y_train.rename("spam")).agg(["mean", "median"]).round(2).T)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, ["length", "n_words", "n_digits", "n_upper"]):
    for lab, name in [(0, "ham"), (1, "spam")]:
        ax.hist(F_train.loc[y_train == lab, col], bins=40, alpha=0.6, density=True, label=name)
    ax.set_title(col)
axes[0].legend()
fig.tight_layout()
plt.show()

In [ ]:
# 每一類最常見的詞（用 CountVectorizer 的預設斷詞；這裡只是看，第 2 節才正式建詞彙表）
# 觀察：（兩類最常見的詞有哪些一樣？那些詞能幫忙分辨嗎？spam 的前幾名有什麼共同點？
#        「在 spam 出現的比例 ÷ 在 ham 出現的比例」最大的詞是哪些？它們在 ham 裡出現過幾次？）
from sklearn.feature_extraction.text import CountVectorizer

cv_eda = CountVectorizer().fit(X_train)
words = cv_eda.get_feature_names_out()
C = cv_eda.transform(X_train)
cnt = pd.DataFrame(
    {"ham": np.asarray(C[(y_train == 0).to_numpy()].sum(axis=0)).ravel(), "spam": np.asarray(C[(y_train == 1).to_numpy()].sum(axis=0)).ravel()},
    index=words,
)
print("詞彙表大小：", len(words))
top = pd.concat({k: cnt[k].sort_values(ascending=False).head(15).reset_index() for k in ["ham", "spam"]}, axis=1)
display(top)

rate = cnt / cnt.sum()  # 每一類裡，這個詞佔所有詞的比例
ratio = ((rate["spam"] + 1e-9) / (rate["ham"] + 1e-9)).sort_values()
print("spam / ham 比例最大的 10 個詞（出現 ≥ 10 次）：")
display(cnt.loc[ratio[cnt.loc[ratio.index].sum(axis=1) >= 10].index[-10:]].assign(ratio=ratio).round(1))
print("只出現 1 次的詞：", int((cnt.sum(axis=1) == 1).sum()), f"（{(cnt.sum(axis=1) == 1).mean():.0%}）")

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 類別不平衡：
#     (a) spam 佔多少？「全部猜 ham」的 accuracy 是多少？
#     (b) 對垃圾簡訊過濾器來說，「把正常簡訊丟進垃圾桶」（false positive）和「漏掉一則垃圾簡訊」（false negative），
#         哪一個比較嚴重？所以本章要看 spam 類的哪個指標（precision、recall）？（對照第三章的不平衡資料）
# 答：
#
# (2) 重複的簡訊：有些 spam 一字不差地出現很多次，而且 train 和 test 都有。
#     這會讓測試成績偏高還是偏低？要不要先去除重複？兩種做法各回答「測試集在模擬什麼情境」。
# 答：
#
# (3) 詞頻：
#     (a) 只出現 1 次的詞佔詞彙表多少？
#     (b) 先預測：一個「只在 spam 出現過、ham 裡 0 次」的詞，如果直接用比例估計 P(詞 | ham) = 0，
#         一則 ham 簡訊只要含有這個詞，P(簡訊 | ham) 會變成多少？（3D 的實驗 1 會驗證）
# 答：
#
# (4) 數值特徵：length、n_digits 在 spam 與 ham 的分布像常態分布嗎？
#     先預測：只用這 6 個特徵的 Gaussian NB，會比用詞的 Multinomial NB 好還是差？
# 答：

---
## 2. 前處理與 Baseline

#### 前處理：詞袋（bag of words）

`CountVectorizer`（預設設定）只用訓練集 fit：把每則簡訊轉小寫、切成詞（2 個字元以上的英數字），建一個詞彙表，再把每則簡訊變成長度 $V$ 的詞頻向量。**詞的順序被丟掉了**（所以叫「袋子」）："free call now" 和 "call now free" 是同一個向量。

測試集出現、訓練集沒看過的詞，會被直接忽略（詞彙表裡沒有它的欄位）。

#### Baseline

- **Dummy prior**：永遠輸出訓練集的 spam 比例。
- **關鍵字規則**：含有 5 位以上的連續數字（電話號碼、簡碼）**或**含 "free"、"txt"、"claim"、"prize"、"won" 其中一個詞 → spam。這是人工寫的規則，看「不用學習」能做到哪裡。

#### 本節名詞

- **precision**（spam 類）：被判成 spam 的訊息裡，真的是 spam 的比例 —— 越高越不會誤殺正常簡訊。
- **recall**（spam 類）：真的 spam 裡，被抓到的比例。
- **F1**：precision 與 recall 的調和平均 $\frac{2PR}{P + R}$；兩者有一個很低，F1 就很低。
- **average precision（AP）**：PR 曲線底下的面積，用機率排序，跟門檻無關；不平衡資料比 ROC AUC 更敏感。
- 5-fold 分層 CV，報告「平均 ± 標準差」（同前幾章）。

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

# 整章共用的 5-fold（分層、打亂、固定 seed）
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

vec = CountVectorizer().fit(X_train)
Xtr = vec.transform(X_train)  # (n, V) 稀疏
Xte = vec.transform(X_test)
vocab = vec.get_feature_names_out()
ytr, yte = y_train.to_numpy(), y_test.to_numpy()
print(f"詞彙表大小 V = {len(vocab)}；Xtr {Xtr.shape}，非零比例 {Xtr.nnz / np.prod(Xtr.shape):.4%}，每則平均 {Xtr.nnz / Xtr.shape[0]:.1f} 個不同的詞")
print("測試集中「詞彙表裡沒有」而被丟掉的詞次數比例：",
      round(1 - Xte.sum() / CountVectorizer(vocabulary=CountVectorizer().fit(X_test).vocabulary_).fit_transform(X_test).sum(), 4))


class KeywordRule(ClassifierMixin, BaseEstimator):
    """人工規則：含 5 位以上連續數字、或含任一關鍵字 → spam。輸入是文字的 Series。"""

    keywords = ("free", "txt", "claim", "prize", "won")

    def fit(self, X: pd.Series, y: pd.Series) -> "KeywordRule":
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X: pd.Series) -> np.ndarray:
        s = pd.Series(X).str.lower()
        hit = s.str.contains(r"\d{5,}")
        for w in self.keywords:
            hit |= s.str.contains(rf"\b{w}\b")
        return hit.astype(int).to_numpy()


SCORING = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}


def cv_summary(model: BaseEstimator, X=X_train, y: pd.Series = y_train) -> dict[str, str]:
    """5-fold CV。回傳各指標的「平均 ± 標準差」；有 predict_proba 的模型另外算 average precision。"""
    scoring = dict(SCORING)
    if hasattr(model, "predict_proba"):
        scoring["AP"] = "average_precision"
    s = cross_validate(model, X, y, cv=cv, scoring=scoring)
    out = {k: f"{s['test_' + k].mean():.4f} ± {s['test_' + k].std():.4f}" for k in scoring}
    out.setdefault("AP", "—")
    return out


import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Dummy 永遠不猜 spam → precision 未定義的警告
    baseline = pd.DataFrame(
        {
            "Dummy prior": cv_summary(DummyClassifier(strategy="prior")),
            "Keyword rule": cv_summary(KeywordRule()),
        }
    ).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) Dummy prior 的 accuracy 很高，precision、recall、F1 卻都是 0。為什麼？這說明本章為什麼不用 accuracy 選模型？
# 答：
#
# (2) 關鍵字規則：
#     (a) precision 和 recall 哪一個比較高？這代表它的錯誤主要是「誤殺」還是「漏抓」？
#     (b) 想提高 recall，最直接的做法是加更多關鍵字。這會讓 precision 怎麼變？為什麼？
#     (c) 「手寫規則」的根本限制是什麼？（想想：要寫幾條規則？每個詞要多「可疑」才算？兩個弱線索加起來呢？）
#         樸素貝氏怎麼把「每個詞各自有多可疑」變成數字、再加起來？（先猜，3A 會看到）
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | Multinomial NB：數數 → 平滑 → log 空間打分數 → 換成機率 | 3、4、5、6 |
| 3B | Bernoulli NB：只看「有沒有出現」；稀疏矩陣上的小技巧 | 8 |
| 3C | Gaussian NB：連續特徵 | 9 |
| 3D | 實驗：平滑、underflow、獨立假設、生成式 vs 判別式 | — |

### 3A. Multinomial Naive Bayes

**模型（生成故事）**：要產生一則第 $k$ 類的簡訊，就從「第 $k$ 類的詞袋」裡**抽後放回**地抽詞，每次抽到第 $j$ 個詞的機率是 $\theta_{kj}$（$\sum_j \theta_{kj} = 1$）。抽到的每個詞互相獨立（這就是「樸素」的假設）。所以

$$P(x \mid y = k) \propto \prod_{j=1}^{V} \theta_{kj}^{\,x_{ij}} \quad\Rightarrow\quad \log P(x_i \mid y = k) = \sum_{j} x_{ij} \log \theta_{kj} + \text{常數}$$

（常數是多項式係數，對每一類都一樣，比大小時可以丟掉；sklearn 也丟掉。）

**訓練 = 數數**（前置複習 7 的 MLE，再加上平滑）：

$$\pi_k = \frac{n_k}{n}, \qquad \theta_{kj} = \frac{N_{kj} + \alpha}{\sum_{l} N_{kl} + \alpha V}$$

- $n_k$：第 $k$ 類有幾則簡訊；$N_{kj}$：第 $k$ 類的所有簡訊裡，第 $j$ 個詞**總共**出現幾次。
- $\alpha$：每個詞先假裝多看到 $\alpha$ 次（$\alpha = 1$：Laplace 平滑）。分母加 $\alpha V$ 是為了讓 $\sum_j \theta_{kj}$ 仍然 = 1。

**預測 = 加分數**：

$$\text{JLL}_{ik} = \log \pi_k + \sum_j x_{ij} \log \theta_{kj}, \qquad P(y = k \mid x_i) = \frac{e^{\text{JLL}_{ik}}}{\sum_{k'} e^{\text{JLL}_{ik'}}}$$

每出現一次第 $j$ 個詞，第 $k$ 類就加 $\log \theta_{kj}$ 分（負數；越常見的詞扣越少）。兩類的分數差 $\sum_j x_{ij} (\log \theta_{1j} - \log \theta_{0j})$ —— 每個詞貢獻一個「可疑度」，加起來（TODO 2 (2)(c) 的答案）。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 詞頻矩陣（稀疏） | $(n, V)$ | `X` |
| $Y$ | 標籤的 one-hot | $(n, K)$ | `Y = np.eye(K)[y_idx]` |
| $n_k$ | 每一類的簡訊數 | $(K,)$ | `self.class_count_` |
| $N_{kj}$ | 每一類、每個詞的總次數 | $(K, V)$ | `self.feature_count_` |
| $\log \pi_k$ | log 先驗 | $(K,)$ | `self.class_log_prior_` |
| $\log \theta_{kj}$ | log 詞機率 | $(K, V)$ | `self.feature_log_prob_` |
| JLL | 每一筆、每一類的分數 | $(n_\text{query}, K)$ | `jll` |
| proba | $P(y = k \mid x)$ | $(n_\text{query}, K)$ | `proba` |

（屬性名稱跟 sklearn `MultinomialNB` 一樣，方便對照。）

**下標公式**：

- TODO 3：$N_{kj} = \sum_i Y_{ik} X_{ij}$ —— 加總 $i$（簡訊），保留 $k$、$j$ → `Y.T @ X`（前置複習 11）。
- TODO 4：$\log \theta_{kj} = \log(N_{kj} + \alpha) - \log \sum_l (N_{kl} + \alpha)$ —— 分母對 $l$ 加總、保留 $k$ → `.sum(axis=1, keepdims=True)`。
- TODO 5：$\text{JLL}_{ik} = \sum_j X_{ij} \log\theta_{kj} + \log \pi_k$ —— 加總 $j$ → `X @ feature_log_prob_.T`，再加 $(K,)$ 的先驗（廣播）。
- TODO 6：$P_{ik} = \exp(\text{JLL}_{ik} - \log\sum_{k'} e^{\text{JLL}_{ik'}})$ —— 每一列做 log-sum-exp（前置複習 6）。

#### 最小例子：TODO 3、4（數數與平滑）

4 則簡訊、詞彙表 3 個詞 `["free", "hi", "win"]`、$\alpha = 1$：

| 簡訊 | free | hi | win | 類別 |
| --- | --- | --- | --- | --- |
| 0 | 1 | 0 | 1 | spam (1) |
| 1 | 0 | 2 | 0 | ham (0) |
| 2 | 2 | 0 | 0 | spam (1) |
| 3 | 0 | 1 | 0 | ham (0) |

1. **正確答案**：兩層迴圈數 $N_{kj}$，再用公式算 $\theta$。目標 $N = \begin{pmatrix}0 & 3 & 0\\ 3 & 0 & 1\end{pmatrix}$、$\theta_\text{ham} = (1/6, 4/6, 1/6)$、$\theta_\text{spam} = (4/7, 1/7, 2/7)$。
2. **積木**：one-hot `Y`、`Y.T` 的 shape、`N + alpha`、`.sum(axis=1, keepdims=True)`。
3. **留給你**：`Y.T @ X` 的意思；log 的分子分母怎麼組。

In [ ]:
# 最小例子：TODO 3、4（數數與平滑）
X_mini = np.array([[1, 0, 1], [0, 2, 0], [2, 0, 0], [0, 1, 0]], dtype=float)  # (n=4, V=3)
y_mini = np.array([1, 0, 1, 0])
K_mini, alpha_mini = 2, 1.0

# ---- 1. 正確答案：迴圈 ----
N_loop = np.zeros((K_mini, 3))
for i_mini in range(4):
    for j_mini in range(3):
        N_loop[y_mini[i_mini], j_mini] += X_mini[i_mini, j_mini]
theta_loop = np.array([[(N_loop[k, j] + alpha_mini) / (N_loop[k].sum() + alpha_mini * 3) for j in range(3)] for k in range(K_mini)])
print("正確答案 N =\n", N_loop)
print("θ =\n", theta_loop.round(4), " 每列總和", theta_loop.sum(axis=1))

# ---- 2. 積木 ----
Y_mini = np.eye(K_mini)[y_mini]
print("\nY = np.eye(K)[y]", Y_mini.shape, "\n", Y_mini)
print("Y.T            ", Y_mini.T.shape, "← (K, n)，乘上 (n, V) 的 X 會得到 (K, V)")
print("N + alpha      \n", N_loop + alpha_mini)
print("(N + alpha).sum(axis=1, keepdims=True)", (N_loop + alpha_mini).sum(axis=1, keepdims=True).shape, (N_loop + alpha_mini).sum(axis=1, keepdims=True).ravel(),
      "← = 每類總詞數 + αV")

# ---- 3. 留給你 ----
# 問題：Y.T @ X 的第 (k, j) 格是什麼的總和？log θ = log(分子) − log(分母)：分子、分母各是上面哪個積木？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# m_mini = MultinomialNaiveBayes(alpha=1.0).fit(X_mini, y_mini); print(m_mini.feature_count_, np.exp(m_mini.feature_log_prob_).round(4))

#### 最小例子：TODO 5、6（打分數與換成機率）

接上一格的 $\log\theta$、$\log\pi = (\log 0.5, \log 0.5)$。查詢兩則：`q0 = "free free win"` → $(2, 0, 1)$；`q1` 是一則很長的 spam，「free」出現 2,000 次 → $(2000, 0, 0)$。

1. **正確答案**：逐類用公式加總；機率用「先 exp 再正規化」。目標 q0：$P(\text{spam}) \approx 0.9527$。q1 用天真的做法會出問題。
2. **積木**：`X @ log_theta.T` 的 shape、加上 `log_prior` 的廣播、`np.exp` 的 underflow、`logsumexp`。
3. **留給你**：用 log-sum-exp 正規化。

In [ ]:
# 最小例子：TODO 5、6（打分數與換成機率）
from scipy.special import logsumexp

log_theta_mini = np.log(theta_loop)  # (K, V)
log_prior_mini = np.log([0.5, 0.5])  # (K,)
Q_mini = np.array([[2.0, 0.0, 1.0], [2000.0, 0.0, 0.0]])  # (n_query=2, V=3)

# ---- 1. 正確答案：逐類加總（q0），再天真地 exp ----
jll_loop = np.array([[log_prior_mini[k] + sum(Q_mini[i, j] * log_theta_mini[k, j] for j in range(3)) for k in range(2)] for i in range(2)])
print("JLL（迴圈）=\n", jll_loop.round(3))
with np.errstate(invalid="ignore"):
    naive = np.exp(jll_loop) / np.exp(jll_loop).sum(axis=1, keepdims=True)
print("天真的 exp / sum：\n", naive.round(4), "← q0 正確（spam 0.9527）；q1 是 nan")

# ---- 2. 積木 ----
print("\nQ @ log_theta.T          ", (Q_mini @ log_theta_mini.T).shape, "← (n_query, K)")
print("+ log_prior（廣播 (K,)）  ", (Q_mini @ log_theta_mini.T + log_prior_mini).round(3).tolist())
print("np.exp(JLL) of q1        ", np.exp(jll_loop[1]), "← 兩類都 underflow 成 0 → 0/0 = nan")
print("logsumexp(JLL, axis=1, keepdims=True)", logsumexp(jll_loop, axis=1, keepdims=True).shape, logsumexp(jll_loop, axis=1, keepdims=True).ravel().round(3))
print("JLL − 每列最大值         ", (jll_loop - jll_loop.max(axis=1, keepdims=True)).round(3).tolist(), "← 最大的那一類變成 0")

# ---- 3. 留給你 ----
# 問題：log P(y=k|x) = JLL − logsumexp(JLL)（每一列）；再 exp。q1 的 spam 機率應該非常接近 1 而不是 nan。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(m_mini.predict_proba(Q_mini).round(6))

In [ ]:
from typing import Self

from numpy.typing import ArrayLike
from scipy import sparse


class MultinomialNaiveBayes:
    """Multinomial NB。X 是詞頻（ndarray 或 scipy 稀疏矩陣）。屬性名稱仿 sklearn MultinomialNB。"""

    def __init__(self, alpha: float = 1.0) -> None:
        self.alpha = alpha

    def _count(self, X, y: ArrayLike) -> None:
        """存下 classes_、class_count_ (K,)、feature_count_ (K, V)。"""
        self.classes_, y_idx = np.unique(np.asarray(y), return_inverse=True)
        Y = np.eye(len(self.classes_))[y_idx]  # (n, K)
        self.class_count_ = Y.sum(axis=0)
        # TODO 3（程式）：每一類、每個詞的總次數 feature_count_ (K, V)
        #   - 下標公式 N[k, j] = Σ_i Y[i, k] · X[i, j] → 一次矩陣乘法
        #   - X 可能是稀疏矩陣：ndarray @ 稀疏矩陣 會回傳 ndarray，外面再包 np.asarray(...) 保險
        raise NotImplementedError("TODO 3：feature_count_")

    def fit(self, X, y: ArrayLike) -> Self:
        self._count(X, y)
        self.class_log_prior_ = np.log(self.class_count_ / self.class_count_.sum())
        smoothed = self.feature_count_ + self.alpha  # (K, V)
        # TODO 4（程式）：log θ (K, V)
        #   - log θ[k, j] = log(N[k, j] + α) − log(Σ_l (N[k, l] + α))
        #   - 分母每一類一個數：對「詞」那一軸加總，keepdims=True 才能跟 (K, V) 廣播
        raise NotImplementedError("TODO 4：feature_log_prob_")
        return self

    def joint_log_likelihood(self, X) -> np.ndarray:
        """X (n_query, V) → JLL (n_query, K) = log π_k + Σ_j x_j log θ_kj。"""
        # TODO 5（程式）：JLL
        #   - X @ self.feature_log_prob_.T 是 (n_query, K)；再加上 (K,) 的 class_log_prior_（廣播）
        #   - 稀疏矩陣 @ ndarray 回傳 ndarray（某些版本是 np.matrix → 用 np.asarray 包起來）
        raise NotImplementedError("TODO 5：joint_log_likelihood")

    def predict_log_proba(self, X) -> np.ndarray:
        """(n_query, K)：log P(y = k | x)。"""
        jll = self.joint_log_likelihood(X)
        # TODO 6（程式）：log-sum-exp 正規化
        #   - log P = JLL − logsumexp(JLL, axis=1, keepdims=True)
        #   - 不要先 np.exp(jll) 再除以總和：長簡訊的 JLL 是 −1000 這種數，exp 之後兩類都變 0（最小例子 q1）
        raise NotImplementedError("TODO 6：log-sum-exp")

    def predict_proba(self, X) -> np.ndarray:
        return np.exp(self.predict_log_proba(X))

    def predict(self, X) -> np.ndarray:
        """比大小只需要 JLL（分母 P(x) 對每一類都一樣）。"""
        return self.classes_[self.joint_log_likelihood(X).argmax(axis=1)]

In [ ]:
from sklearn.naive_bayes import BernoulliNB, GaussianNB, MultinomialNB

mnb_scratch = MultinomialNaiveBayes(alpha=1.0).fit(Xtr, ytr)
mnb_sk = MultinomialNB(alpha=1.0).fit(Xtr, ytr)
print("✅ feature_count_ 與 sklearn 相同：", np.array_equal(mnb_scratch.feature_count_, mnb_sk.feature_count_))
print("✅ feature_log_prob_ 最大差異：", np.abs(mnb_scratch.feature_log_prob_ - mnb_sk.feature_log_prob_).max(), "（應 < 1e-12）")
print("✅ 每一類 Σ_j θ_kj：", np.exp(mnb_scratch.feature_log_prob_).sum(axis=1), "（應都 = 1）")
print("✅ predict_proba 最大差異：", np.abs(mnb_scratch.predict_proba(Xte) - mnb_sk.predict_proba(Xte)).max(), "（應 < 1e-10）")
print("✅ predict 一致比例：", np.mean(mnb_scratch.predict(Xte) == mnb_sk.predict(Xte)), "（應 = 1.0）")
print("   有沒有 nan：", np.isnan(mnb_scratch.predict_proba(Xte)).any(), "（應 False）")

# 一則簡訊的「分數明細」：每個詞貢獻多少 log(θ_spam / θ_ham)
def explain(text: str, model: MultinomialNaiveBayes = mnb_scratch) -> pd.DataFrame:
    """一則文字 → 每個詞的次數與 log 比值貢獻，加上先驗；最後一列是總和（= JLL_spam − JLL_ham）。"""
    x = vec.transform([text])
    idx = x.indices
    log_ratio = model.feature_log_prob_[1] - model.feature_log_prob_[0]
    rows = pd.DataFrame({"word": vocab[idx], "count": x.data, "log θ_spam − log θ_ham": log_ratio[idx]})
    rows["contribution"] = rows["count"] * rows["log θ_spam − log θ_ham"]
    rows = rows.sort_values("contribution")
    prior = model.class_log_prior_[1] - model.class_log_prior_[0]
    extra = pd.DataFrame([{"word": "(prior)", "contribution": prior}, {"word": "TOTAL", "contribution": rows["contribution"].sum() + prior}])
    return pd.concat([rows, extra], ignore_index=True)


example = "WINNER!! You have won a FREE prize. Call 09061701461 now to claim"
display(explain(example).round(3))
jll_ex = mnb_scratch.joint_log_likelihood(vec.transform([example]))[0]
print("JLL (ham, spam) =", jll_ex.round(3), " P(spam) =", mnb_scratch.predict_proba(vec.transform([example]))[0, 1].round(6))

In [ ]:
# TODO 7（問答）手算與解讀
#
# (1) 用最小例子的 θ（ham (1/6, 4/6, 1/6)、spam (4/7, 1/7, 2/7)）、先驗各 0.5，手算 q0 = "free free win" 的
#     (a) 兩類的 JLL（寫出算式，用自然對數）；
#     (b) P(spam | q0)。跟最小例子的 0.9527 一致嗎？
# 答：
#
# (2) 看 explain 的表：
#     (a) 哪幾個詞把分數往 spam 推得最多？有沒有往 ham 推的詞？
#     (b) TOTAL 是什麼？它跟 P(spam) 的關係是 P(spam) = σ(TOTAL)（σ 是第三章的 sigmoid）—— 為什麼？
#         （提示：二元時 P(spam|x) = e^{JLL_1} / (e^{JLL_0} + e^{JLL_1})，分子分母同除 e^{JLL_1}）
#     (c) 所以二元的 Multinomial NB 也是一個「線性模型」：w_j = ?、b = ?。
#         跟第三章的邏輯回歸「長得一樣」，差在哪裡？（提示：w 是怎麼得到的 —— 數數，還是最小化 log loss？）
# 答：
#
# (3) 「fit 只是數數」：
#     (a) fit 的成本是 O(?)（用 X 的非零元素個數 nnz 表示）？跟第三章用梯度下降訓練邏輯回歸比呢？
#     (b) 新來 100 則已標記的簡訊，要更新模型需要重新看一遍舊資料嗎？（提示：feature_count_ 可以怎麼更新）
# 答：

### 3B. Bernoulli Naive Bayes

Multinomial 在乎詞出現**幾次**；Bernoulli 只在乎詞**有沒有出現**，而且「沒出現」也算證據：

$$P(x \mid y = k) = \prod_{j=1}^{V} p_{kj}^{\,b_j}\,(1 - p_{kj})^{1 - b_j}, \qquad b_j = [x_j > 0]$$

$$p_{kj} = \frac{(\text{第 } k \text{ 類中含第 } j \text{ 個詞的簡訊數}) + \alpha}{n_k + 2\alpha}$$

（分母加 $2\alpha$：「有」和「沒有」兩種結果各假裝多看到 $\alpha$ 次。）

**稀疏矩陣上的技巧**（TODO 8）：直接算 $\sum_j [b_j \log p_{kj} + (1 - b_j)\log(1 - p_{kj})]$ 需要 $1 - b$，而 $1 - b$ 幾乎全是 1 —— 一個 $(n, V)$ 的**稠密**矩陣（4,459 × 7,700 ≈ 3,400 萬格）。改寫：

$$\sum_j \big[b_j \log p_{kj} + (1 - b_j) \log(1 - p_{kj})\big] = \sum_j b_j \underbrace{\big[\log p_{kj} - \log(1 - p_{kj})\big]}_{\text{只在有出現的詞上算}} + \underbrace{\sum_j \log(1 - p_{kj})}_{\text{每一類一個常數}}$$

第一項是稀疏矩陣 $B$ 乘一個 $(V, K)$ 矩陣，第二項是 $(K,)$ 的向量。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $B$ | 二值化的 $X$：$B_{ij} = [x_{ij} > 0]$ | $(n, V)$ 稀疏 | `B = (X > 0).astype(float)` |
| $p_{kj}$ | 第 $k$ 類含第 $j$ 個詞的機率 | $(K, V)$ | `p = exp(feature_log_prob_)` |
| $\log(1 - p_{kj})$ | 「沒出現」的 log 機率 | $(K, V)$ | `log_neg = np.log1p(-p)` |
| JLL | 同 3A | $(n_\text{query}, K)$ | — |

**$V = 2$ 的手算例**：$p_\text{spam} = (0.5, 0.2)$、查詢只含第 0 個詞 $b = (1, 0)$ → 直接算 $\log 0.5 + \log 0.8$；改寫 $1 \cdot (\log 0.5 - \log 0.5) + (\log 0.5 + \log 0.8)$ ✓（$= \log 0.4$）。

#### 最小例子：TODO 8（不用稠密的 1 − B）

接 3A 最小例子的資料；`p` 用公式算好了。查詢 `q = "free"` → $b = (1, 0, 0)$。

1. **正確答案**：逐詞算「有 → $\log p$、沒有 → $\log(1-p)$」再加總。
2. **積木**：`B @ (log_p − log_neg).T`、`log_neg.sum(axis=1)`；以及 $1 - B$ 對稀疏矩陣做會發生什麼。
3. **留給你**：兩個積木加上先驗。

In [ ]:
# 最小例子：TODO 8（不用稠密的 1 − B）
B_mini = (X_mini > 0).astype(float)
doc_count_mini = np.eye(2)[y_mini].T @ B_mini  # (K, V)：每類含這個詞的簡訊數
p_mini = (doc_count_mini + 1.0) / (np.bincount(y_mini)[:, None] + 2.0)
log_p_mini, log_neg_mini = np.log(p_mini), np.log1p(-p_mini)
q_mini = np.array([[1.0, 0.0, 0.0]])

# ---- 1. 正確答案：逐詞 ----
for k in range(2):
    s = sum(log_p_mini[k, j] if q_mini[0, j] > 0 else log_neg_mini[k, j] for j in range(3))
    print(f"類別 {k}：p = {p_mini[k].round(3)}  Σ = {s:.4f}（再加 log 先驗 {np.log(0.5):.4f}）")

# ---- 2. 積木 ----
print("\nq @ (log_p − log_neg).T", (q_mini @ (log_p_mini - log_neg_mini).T).round(4), "← 只在有出現的詞上算")
print("log_neg.sum(axis=1)    ", log_neg_mini.sum(axis=1).round(4), "← 每類一個常數")
B_big = sparse.random(2000, 7000, density=0.002, format="csr", random_state=0)
print(f"稀疏 B (2000×7000) 存 {B_big.nnz} 個非零；1 − B 會變成 {type(1 - B_big.toarray()).__name__}，{2000 * 7000 * 8 / 1e6:.0f} MB 的稠密矩陣")

# ---- 3. 留給你 ----
# 問題：JLL = (哪個積木) + (哪個積木) + class_log_prior_？加起來應該等於上面迴圈的 Σ + log 先驗。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(BernoulliNaiveBayes().fit(X_mini, y_mini).joint_log_likelihood(q_mini).round(4))

In [ ]:
class BernoulliNaiveBayes(MultinomialNaiveBayes):
    """Bernoulli NB：先把 X 二值化（> 0 → 1）。沿用 Multinomial 的 _count、predict_log_proba、predict。"""

    @staticmethod
    def _binarize(X):
        return (X > 0).astype(float)

    def fit(self, X, y: ArrayLike) -> Self:
        self._count(self._binarize(X), y)  # feature_count_ 變成「含這個詞的簡訊數」
        self.class_log_prior_ = np.log(self.class_count_ / self.class_count_.sum())
        p = (self.feature_count_ + self.alpha) / (self.class_count_[:, None] + 2 * self.alpha)  # (K, V)
        self.feature_log_prob_ = np.log(p)
        self.neg_log_prob_ = np.log1p(-p)  # log(1 − p)
        return self

    def joint_log_likelihood(self, X) -> np.ndarray:
        B = self._binarize(X)
        # TODO 8（程式）：Bernoulli 的 JLL，不建立稠密的 1 − B
        #   - B @ (feature_log_prob_ − neg_log_prob_).T  → (n_query, K)
        #   - 加上 neg_log_prob_ 對「詞」那一軸的總和 (K,)，再加 class_log_prior_ (K,)
        raise NotImplementedError("TODO 8：Bernoulli JLL")


bnb_scratch = BernoulliNaiveBayes(alpha=1.0).fit(Xtr, ytr)
bnb_sk = BernoulliNB(alpha=1.0).fit(Xtr, ytr)
print("✅ feature_log_prob_ 最大差異：", np.abs(bnb_scratch.feature_log_prob_ - bnb_sk.feature_log_prob_).max(), "（應 < 1e-12）")
print("✅ predict_proba 最大差異：", np.abs(bnb_scratch.predict_proba(Xte) - bnb_sk.predict_proba(Xte)).max(), "（應 < 1e-10）")
print("✅ predict 一致比例：", np.mean(bnb_scratch.predict(Xte) == bnb_sk.predict(Xte)), "（應 = 1.0）")

### 3C. Gaussian Naive Bayes

特徵是連續值時（第 1 節的 6 個數值特徵：長度、數字個數……），假設**每一類的每個特徵各自是常態分布**，而且（樸素）彼此獨立：

$$\log P(x \mid y = k) = \sum_{j=1}^{d} \left[-\frac{1}{2}\log(2\pi\sigma_{kj}^2) - \frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^2}\right]$$

訓練還是數數：$\mu_{kj}$ = 第 $k$ 類第 $j$ 個特徵的平均，$\sigma^2_{kj}$ = 變異數（除以 $n_k$）。
為了避免某一類某個特徵的變異數剛好是 0（例如 ham 的 `has_url` 全是 0 → 除以 0），sklearn 會加一個很小的 $\epsilon$ = `var_smoothing`（預設 $10^{-9}$）× 所有特徵中最大的變異數。

#### 符號表（3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $F$ | 數值特徵 | $(n, d)$，$d = 6$ | `F` |
| $\mu_{kj}$ | 每類每個特徵的平均 | $(K, d)$ | `self.theta_` |
| $\sigma^2_{kj}$ | 每類每個特徵的變異數（含 $\epsilon$） | $(K, d)$ | `self.var_` |
| $F_{i}$ 與 $\mu_k$ 的差 | 每一筆、每一類、每個特徵 | $(n_\text{query}, K, d)$ | `F[:, None, :] - self.theta_[None, :, :]` |

**下標公式**（TODO 9）：$\text{JLL}_{ik} = \log\pi_k + \sum_j \left[-\frac{1}{2}\log(2\pi\sigma^2_{kj}) - \frac{(F_{ij} - \mu_{kj})^2}{2\sigma^2_{kj}}\right]$ —— 保留 $i$、$k$，加總 $j$。三個下標同時出現 → 用 `[:, None, :]`／`[None, :, :]` 造出 $(n, K, d)$，再 `.sum(axis=-1)`。這裡 $d = 6$ 很小，三維暫存陣列沒問題（對照第四章：$d$ 大時要避免）。

#### 最小例子：TODO 9（Gaussian 的 log 密度）

$K = 2$、$d = 2$：$\mu = \begin{pmatrix}0 & 10\\ 5 & 20\end{pmatrix}$、$\sigma^2 = \begin{pmatrix}1 & 4\\ 1 & 4\end{pmatrix}$、先驗各 0.5。查詢 $F = (1, 12)$。

1. **正確答案**：兩層迴圈，用 `scipy.stats.norm.logpdf` 逐項加總。
2. **積木**：`F[:, None, :] − mu[None, :, :]` 的 shape、除以 `var[None]`、`np.log(2π var)`。
3. **留給你**：組成公式、沿哪一軸加總。

In [ ]:
# 最小例子：TODO 9（Gaussian 的 log 密度）
from scipy.stats import norm

mu_mini = np.array([[0.0, 10.0], [5.0, 20.0]])  # (K, d)
var_mini = np.array([[1.0, 4.0], [1.0, 4.0]])
F_mini = np.array([[1.0, 12.0]])  # (n_query=1, d=2)

# ---- 1. 正確答案：迴圈 ----
for k in range(2):
    s = np.log(0.5) + sum(norm.logpdf(F_mini[0, j], mu_mini[k, j], np.sqrt(var_mini[k, j])) for j in range(2))
    print(f"類別 {k}：JLL = {s:.4f}")

# ---- 2. 積木 ----
diff_mini = F_mini[:, None, :] - mu_mini[None, :, :]
print("\nF[:, None, :] − mu[None, :, :]", diff_mini.shape, "← (n_query, K, d)\n", diff_mini)
print("diff² / var[None]              ", (diff_mini**2 / var_mini[None]).round(3).tolist())
print("np.log(2π var)                 ", np.log(2 * np.pi * var_mini).round(4).tolist(), "← (K, d)，要加 [None] 才能跟 (n, K, d) 對齊")

# ---- 3. 留給你 ----
# 問題：−½·log(2πσ²) − ½·diff²/σ²，沿哪一軸加總才會剩 (n_query, K)？最後別忘了先驗。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# g_mini = GaussianNaiveBayes(); g_mini.theta_, g_mini.var_, g_mini.class_log_prior_ = mu_mini, var_mini, np.log([0.5, 0.5]); print(g_mini.joint_log_likelihood(F_mini).round(4))

In [ ]:
class GaussianNaiveBayes(MultinomialNaiveBayes):
    """Gaussian NB：每類每個特徵一個常態分布。沿用 predict_log_proba、predict。"""

    def __init__(self, var_smoothing: float = 1e-9) -> None:
        self.var_smoothing = var_smoothing

    def fit(self, F: ArrayLike, y: ArrayLike) -> Self:
        F = np.asarray(F, dtype=float)
        self.classes_, y_idx = np.unique(np.asarray(y), return_inverse=True)
        K = len(self.classes_)
        self.class_count_ = np.bincount(y_idx, minlength=K).astype(float)
        self.class_log_prior_ = np.log(self.class_count_ / self.class_count_.sum())
        self.epsilon_ = self.var_smoothing * F.var(axis=0).max()
        self.theta_ = np.array([F[y_idx == k].mean(axis=0) for k in range(K)])  # (K, d)
        self.var_ = np.array([F[y_idx == k].var(axis=0) for k in range(K)]) + self.epsilon_
        return self

    def joint_log_likelihood(self, F: ArrayLike) -> np.ndarray:
        F = np.asarray(F, dtype=float)
        # TODO 9（程式）：Gaussian 的 JLL (n_query, K)
        #   - diff = F[:, None, :] − self.theta_[None, :, :]  → (n_query, K, d)
        #   - 每項 −½·log(2π·var) − ½·diff²/var，var 要 [None] 對齊；對 d 那一軸加總；再加 class_log_prior_
        raise NotImplementedError("TODO 9：Gaussian JLL")


Ftr, Fte = F_train.to_numpy(), F_test.to_numpy()
gnb_scratch = GaussianNaiveBayes().fit(Ftr, ytr)
gnb_sk = GaussianNB().fit(Ftr, ytr)
print("✅ theta_、var_ 最大差異：", np.abs(gnb_scratch.theta_ - gnb_sk.theta_).max(), np.abs(gnb_scratch.var_ - gnb_sk.var_).max(), "（應 < 1e-9）")
print("✅ predict_proba 最大差異：", np.abs(gnb_scratch.predict_proba(Fte) - gnb_sk.predict_proba(Fte)).max(), "（應 < 1e-10）")
print("✅ predict 一致比例：", np.mean(gnb_scratch.predict(Fte) == gnb_sk.predict(Fte)), "（應 = 1.0）")

In [ ]:
# 三種 NB 的 5-fold CV（Gaussian 用 6 個數值特徵；另外 Gaussian 用 log1p 轉換後的特徵再試一次）
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer

nb_cmp = {
    "Multinomial NB (words)": (make_pipeline(CountVectorizer(), MultinomialNB()), X_train),
    "Bernoulli NB (words)": (make_pipeline(CountVectorizer(), BernoulliNB()), X_train),
    "Gaussian NB (6 numeric)": (GaussianNB(), F_train),
    "Gaussian NB (log1p of 6 numeric)": (make_pipeline(FunctionTransformer(np.log1p), GaussianNB()), F_train),
}
for name, (model, Xin) in nb_cmp.items():
    baseline.loc[name] = cv_summary(model, Xin)
display(baseline)

In [ ]:
# TODO 10（問答）三種 NB
#
# (1) Multinomial vs Bernoulli：
#     (a) 讀表：兩者的 precision、recall、F1 各差多少？
#     (b) 簡訊很短（每則平均十幾個詞）。Bernoulli 會把「7,700 個詞裡沒出現的那 7,690 個」也當證據。
#         一個在 spam 中出現率 p_spam = 0.3、ham 中 p_ham = 0.01 的詞「沒出現」，貢獻的 log 比值 log(0.7 / 0.99) 是多少？
#         很多這種「沒出現」加起來，會把分數往哪一類推？這對短文字有利還是不利？
#     (c) 一則 spam 裡 "free" 出現 3 次。Multinomial 和 Bernoulli 各怎麼看待這 3 次？
# 答：
#
# (2) Gaussian NB：
#     (a) 只用 6 個數值特徵的 F1 比用詞的差多少？原始特徵與 log1p 之後哪個好？
#     (b) 用 EDA 的分布圖解釋：length、n_digits 在 ham 中像常態分布嗎？ham 的 n_digits 大多是 0、
#         少數很大 —— 用「平均和變異數」描述這種分布，會低估還是高估「很大的值」出現的機率？
#         log1p 之後 precision、recall、AP 各往哪個方向變？為什麼「變得更像常態」不一定讓 F1 變好？
#     (c) has_url 是 0／1 的特徵，用常態分布描述它合理嗎？如果某一類的 has_url 全是 0，σ² 會是多少？var_smoothing 在這裡的作用是什麼？
# 答：

### 3D. 實驗：平滑、underflow、獨立假設、生成式 vs 判別式

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 零頻率問題 | 某個詞在某一類的訓練資料裡 0 次 → $\theta = 0$ → $\log\theta = -\infty$ → 只要出現這個詞，那一類的機率就是 0 |
| underflow | 很多小於 1 的機率相乘，小到 float64 表示不了而變成 0（前置複習 5） |
| 過度自信（overconfident） | 預測機率很極端（接近 0 或 1），但實際上沒那麼常對 |
| 校準（calibration） | 「預測 70% 的那些訊息，真的有 70% 是 spam」；可以把預測依信心分組，比較「平均信心」與「實際猜對的比例」 |
| log loss | 同第三章：$\frac{1}{n}\sum_i -\ln P(\text{正確類別})$；對過度自信的錯誤懲罰很重 |
| 生成式／判別式 | 學 $P(x \mid y)$、$P(y)$ 再反推／直接學 $P(y \mid x)$ |

In [ ]:
# 實驗 1：平滑。α 從 0（不平滑）到很大
from sklearn.metrics import f1_score

rows = []
for a in [0.0, 1e-10, 1e-3, 0.01, 0.1, 0.5, 1.0, 3.0, 10.0]:
    with np.errstate(divide="ignore"):
        m = MultinomialNaiveBayes(alpha=a).fit(Xtr, ytr)
        lp = m.feature_log_prob_
        jll = m.joint_log_likelihood(Xte)
    both_inf = np.isinf(jll).all(axis=1)  # 兩類都 −inf：0/0
    pred = m.classes_[np.nan_to_num(jll, neginf=-1e300).argmax(axis=1)]
    rows.append(
        {
            "alpha": a,
            "log θ 為 −inf 的格數": int(np.isinf(lp).sum()),
            "test 中某一類 JLL = −inf 的筆數": int(np.isinf(jll).any(axis=1).sum()),
            "兩類都 −inf 的筆數": int(both_inf.sum()),
            "test F1": round(f1_score(yte, pred), 4),
        }
    )
display(pd.DataFrame(rows).set_index("alpha"))
print("（這裡只是為了看現象才偷看測試集；選 α 要用 CV，第 5 節）")

# 一個只在 spam 出現過的詞，在不同 α 下的 θ
w = cnt.index[(cnt["ham"] == 0) & (cnt["spam"] >= 5)][0]  # cnt：第 1 節 EDA 的每類詞頻
j = vec.vocabulary_[w]
for a in [0.0, 0.1, 1.0, 10.0]:
    with np.errstate(divide="ignore"):
        th = np.exp(MultinomialNaiveBayes(alpha=a).fit(Xtr, ytr).feature_log_prob_[:, j])
    print(f"α = {a:<5} 詞 '{w}'：θ_ham = {th[0]:.2e}、θ_spam = {th[1]:.2e}、比值 θ_spam/θ_ham = {th[1] / th[0] if th[0] > 0 else np.inf:.1f}")

In [ ]:
# TODO 11（問答）平滑
#
# (1) α = 0 時：
#     (a) 有多少格 log θ = −inf？測試集有幾筆至少一類是 −inf、幾筆兩類都是 −inf？
#     (b) 「兩類都 −inf」是什麼樣的簡訊？（提示：同時含有一個「只在 ham 出現過」和一個「只在 spam 出現過」的詞）
#         這時 predict_proba 會給出什麼？（−inf − (−inf)）
#     (c) 只有一類是 −inf 時：一個詞就能完全否決一個類別，不管其他 20 個詞多像那一類。這合理嗎？
# 答：
#
# (2) α 的意義：
#     (a) 上面那個只在 spam 出現過的詞，α 從 0.1 到 10，θ_spam / θ_ham 怎麼變？α 越大，每個詞的「可疑度」越強還是越弱？
#     (b) α → ∞ 時，所有的 θ_kj 會趨近多少？模型會退化成什麼？（提示：跟第 2 節的 Dummy prior 比）
#     (c) 用偏差—變異說：α 小、α 大各是哪一端？過擬合在 α 的哪一邊？（第 5 節的 validation curve 會驗證）
#     (d) 加 α 等於「每個詞每一類先假裝看過 α 次」。這是一種先驗信念；它跟第三章 L2 正則化的精神有什麼相似？
# 答：

In [ ]:
# 實驗 2：為什麼一定要在 log 空間算？
lens = np.asarray(Xte.sum(axis=1)).ravel()
i_long = int(lens.argmax())
row = Xte[i_long].toarray().ravel()
theta = np.exp(mnb_scratch.feature_log_prob_)
direct = [np.prod(theta[k] ** row) for k in range(2)]
logs = [float(row @ mnb_scratch.feature_log_prob_[k]) for k in range(2)]
print(f"最長的測試簡訊有 {int(lens[i_long])} 個詞：{X_test.iloc[i_long][:120]}...")
print("直接相乘 Π θ^x：", direct)
print("log 空間 Σ x log θ：", np.round(logs, 2), " → 兩類差", round(logs[1] - logs[0], 2))
print("float64 最小的正常數：", np.finfo(float).tiny)

# 幾個詞以後開始 underflow？用「每個詞的 θ 約等於典型值」估計
typical = np.exp(np.median(mnb_scratch.feature_log_prob_[0][np.asarray(Xtr.sum(axis=0)).ravel() > 0]))
print(f"典型詞的 θ ≈ {typical:.1e} → 大約 {np.log(np.finfo(float).tiny) / np.log(typical):.0f} 個詞就會 underflow")

In [ ]:
# 實驗 3：獨立假設錯了會怎樣？把每個詞「複製」r 份（等於每個詞出現的次數都乘 r）
from scipy.sparse import hstack
from sklearn.metrics import log_loss

conf_bins = [0.5, 0.9, 0.99, 0.9999, 1 - 1e-8, 1.0 + 1e-12]
conf_labels = ["0.5–0.9", "0.9–0.99", "0.99–0.9999", "0.9999–(1−1e-8)", "> 1−1e-8"]
fig, ax = plt.subplots(figsize=(7, 3.5))
rows, conf_tbls = [], {}
for r in [1, 2, 5]:
    m = MultinomialNB().fit(hstack([Xtr] * r).tocsr(), ytr)
    proba = m.predict_proba(hstack([Xte] * r).tocsr())[:, 1]
    rows.append(
        {
            "copies r": r,
            "test accuracy": np.mean((proba > 0.5) == yte),
            "機率 < 1e-6 或 > 1 − 1e-6 的比例": np.mean((proba < 1e-6) | (proba > 1 - 1e-6)),
            "test log loss": log_loss(yte, proba),
        }
    )
    ax.hist(proba, bins=np.linspace(0, 1, 21), alpha=0.5, label=f"r={r}", log=True)
    # 信心分組：模型說「有多確定」（max(p, 1 − p)），實際上猜對的比例是多少？
    conf = np.maximum(proba, 1 - proba)
    grp = pd.cut(conf, conf_bins, labels=conf_labels, right=False)
    conf_tbls[f"r={r}"] = pd.DataFrame({"correct": (proba > 0.5) == yte, "conf": conf, "bin": grp}).groupby("bin", observed=False).agg(
        n=("correct", "size"), mean_confidence=("conf", "mean"), actual_accuracy=("correct", "mean")
    )
ax.set_xlabel("predicted P(spam)")
ax.set_title("Distribution of predicted probabilities (log count)")
ax.legend()
plt.show()
display(pd.DataFrame(rows).set_index("copies r").round(4))
print("信心分組：模型的平均信心 vs 實際猜對的比例（測試集）")
display(pd.concat(conf_tbls, axis=1).round(4))

In [ ]:
# TODO 12（問答）underflow 與獨立假設
#
# (1) 實驗 2：
#     (a) 直接相乘的結果是多少？log 空間的兩類差是多少？兩類差這麼大，P(spam) 大約是多少？
#     (b) 典型的詞 θ 約多少？大約幾個詞就會 underflow？簡訊很短，為什麼還是一定要用 log？（想想 email、文章）
#     (c) 3A 的 predict 只用 JLL 比大小，predict_proba 用 log-sum-exp。哪一步是「不得不」的、哪一步只是方便？
# 答：
#
# (2) 實驗 3：複製特徵 = 刻意製造「完全相關」的特徵（樸素假設認為它們是獨立的證據）。
#     (a) accuracy 變了多少？機率極端的比例、log loss 呢？
#     (b) 為什麼複製 r 份之後，JLL 的差（TOTAL）剛好乘 r？P(spam) = σ(r · TOTAL) 會往哪裡跑？
#     (c) 真實的文字裡有沒有「高度相關」的詞？舉兩個例子（提示：片語、同一個 spam 模板裡固定一起出現的詞）。
#         所以原版（r = 1）的機率也會過度自信嗎？看信心分組表 r = 1 的「0.99–0.9999」與「0.9999–(1−1e-8)」兩組：
#         平均信心和實際 accuracy 各是多少？「信心 0.9999」代表一萬則才錯一則，這組 400 則左右實際錯了幾則？r = 5 呢？
#     (d) 結論：獨立假設錯了，「分類」和「機率」哪一個受傷比較大？為什麼分類常常還是對的？
#         （提示：argmax 只要求「對的那一類分數最高」，不在乎高多少）
# 答：

In [ ]:
# 實驗 4：生成式（NB）vs 判別式（邏輯回歸）—— 訓練資料由少到多
from sklearn.linear_model import LogisticRegression

sizes = [30, 60, 100, 200, 400, 800, 1600, len(ytr)]
rows = []
for n_sub in sizes:
    for seed in range(5 if n_sub < len(ytr) else 1):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(ytr), n_sub, replace=False)
        if ytr[idx].sum() < 2:
            continue
        for name, model in [
            ("Multinomial NB", make_pipeline(CountVectorizer(), MultinomialNB())),
            ("Logistic regression", make_pipeline(CountVectorizer(), LogisticRegression(max_iter=2000))),
        ]:
            model.fit(X_train.iloc[idx], ytr[idx])
            rows.append({"n": n_sub, "model": name, "test F1": f1_score(yte, model.predict(X_test))})
gd = pd.DataFrame(rows).groupby(["n", "model"])["test F1"].agg(["mean", "std"]).unstack("model")

fig, ax = plt.subplots(figsize=(7, 4))
for name in ["Multinomial NB", "Logistic regression"]:
    ax.errorbar(gd.index, gd[("mean", name)], yerr=gd[("std", name)], fmt="o-", capsize=3, label=name)
ax.set_xscale("log")
ax.set_xlabel("training samples (log)")
ax.set_ylabel("test F1 (spam)")
ax.set_title("Generative vs discriminative")
ax.legend()
plt.show()
display(gd.round(3))
print("（這個實驗為了看趨勢用了測試集；它不參與任何模型選擇）")

In [ ]:
# TODO 13（問答）生成式 vs 判別式
#
# (1) (a) 訓練資料很少（30–200 則）時，哪一個比較好？差多少？
#     (b) 資料變多之後，差距怎麼變？
# 答：
#
# (2) 為什麼？
#     (a) NB 每個 θ_kj 只靠「數數」就能估計，而且每個詞各自估、互不影響。邏輯回歸的 7,700 個係數要一起最佳化。
#         樣本很少時，哪一個比較不容易被雜訊帶偏？用偏差—變異回答：NB 的「強假設」換到了什麼？
#     (b) 資料很多時，「假設錯了」（獨立假設、詞袋）的代價會顯現。這是偏差還是變異？
#     (c) 對照 TODO 7 (2)(c)：兩者是同一種「線性」形式。差別只在 w 怎麼來 —— 數數（生成式）還是直接最小化分類錯誤（判別式）。
#         （這是 Ng & Jordan (2001) 的經典結論：生成式收斂得快、判別式的極限比較好）
# 答：

---
## 4. 套件實作：sklearn `MultinomialNB`

`CountVectorizer` 和 NB 包進同一個 `Pipeline`：CV 的每個 fold 都只用那個 fold 的訓練部分建詞彙表（否則驗證集的詞會「洩漏」進詞彙表）。

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `TfidfVectorizer` | 詞頻 × idf（越多簡訊都有的詞，權重越低），再把每列正規化成長度 1 |
| `ComplementNB` | 用「**不是**第 $k$ 類」的資料估計參數，對不平衡資料比較穩（Rennie et al. 2003） |
| `binary=True` | `CountVectorizer` 只記有沒有出現（0／1），但模型仍是 Multinomial |
| 最有鑑別力的詞 | $\log\theta_{\text{spam},j} - \log\theta_{\text{ham},j}$ 最大／最小的詞 |

#### 程式結構圖

```
nb_pipe = Pipeline([("vec", CountVectorizer()), ("nb", MultinomialNB(alpha=1.0))])
├── nb_pipe["vec"]：fit 後有 vocabulary_、get_feature_names_out()
└── nb_pipe["nb"] ：fit 後有 class_count_、feature_count_、class_log_prior_、feature_log_prob_
參數名：vec__binary、vec__ngram_range、vec__min_df、nb__alpha
```

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB

nb_pipe = Pipeline([("vec", CountVectorizer()), ("nb", MultinomialNB(alpha=1.0))]).fit(X_train, y_train)
print("✅ 手寫 vs sklearn Pipeline：predict_proba 最大差異",
      np.abs(mnb_scratch.predict_proba(Xte) - nb_pipe.predict_proba(X_test)).max(), "（應 < 1e-10）")

log_ratio = nb_pipe["nb"].feature_log_prob_[1] - nb_pipe["nb"].feature_log_prob_[0]
names = nb_pipe["vec"].get_feature_names_out()
fc = nb_pipe["nb"].feature_count_
disc = pd.DataFrame({"word": names, "log ratio (spam − ham)": log_ratio, "ham count": fc[0], "spam count": fc[1]})
print("最像 spam 的 15 個詞：")
display(disc.sort_values("log ratio (spam − ham)", ascending=False).head(15).round(2).reset_index(drop=True).T)
print("最像 ham 的 15 個詞：")
display(disc.sort_values("log ratio (spam − ham)").head(15).round(2).reset_index(drop=True).T)

variants = {
    "MNB counts": Pipeline([("vec", CountVectorizer()), ("nb", MultinomialNB())]),
    "MNB binary counts": Pipeline([("vec", CountVectorizer(binary=True)), ("nb", MultinomialNB())]),
    "MNB tf-idf": Pipeline([("vec", TfidfVectorizer()), ("nb", MultinomialNB())]),
    "ComplementNB counts": Pipeline([("vec", CountVectorizer()), ("nb", ComplementNB())]),
}
var_tbl = pd.DataFrame({name: cv_summary(m) for name, m in variants.items()}).T
display(var_tbl)

In [ ]:
# TODO 14（問答）
#
# (1) 最像 spam、最像 ham 的詞：
#     (a) 各舉幾個，合理嗎？最像 spam 的詞在 ham 裡的次數大多是多少？
#     (b) 它們的 log ratio 主要是被「spam 次數多」決定，還是被「ham 次數 = 0、只剩 α」決定？
#         α 改大一點，這張排行榜會怎麼變？（TODO 11 (2)(a)）
# 答：
#
# (2) 四種變體：
#     (a) 哪一個 F1 最高？差距跟 std 比起來大嗎？
#     (b) tf-idf 搭配 Multinomial NB 在「理論上」有點怪：Multinomial 的生成故事是「抽詞的次數」，tf-idf 卻不是整數次數。
#         為什麼它還能跑、甚至常常不差？（提示：TODO 7 (2)(c) 的線性形式只需要 x 是非負的數）
#     (c) 在 recall 上，tf-idf 版本和 counts 版本差多少？為什麼 tf-idf（每列正規化成長度 1）會讓 spam 比較難被判成 spam？
#         （提示：先驗 log π 是固定的，詞的總貢獻被縮小了）
# 答：

---
## 5. 調參

全程只用 `X_train`（測試集鎖住）。**選模型的指標用 F1**（spam 類）；第 5 節最後再決定「機率多高才判成 spam」的門檻。

| 類別 | 超參數 | 說明 |
| --- | --- | --- |
| 正則化 | `nb__alpha` | 平滑；越大越簡單（越接近 Dummy），尺度型 → log 尺度搜尋 |
| 特徵 | `vec__binary` | 只記有沒有出現 |
| 特徵 | `vec__ngram_range` | `(1, 2)` = 單詞 + 相鄰兩詞（"free entry"、"call now"）；詞彙表會變大很多 |
| 特徵 | `vec__min_df` | 至少出現在幾則簡訊才收進詞彙表；越大詞彙表越小 |
| 決策 | 門檻 threshold | 不是模型的參數，而是「P(spam) > 多少才判 spam」；用 CV 的預測機率決定 |

#### 本節名詞

- **validation curve**、**grid search**、**learning curve**：同前幾章。
- **`cross_val_predict(..., method="predict_proba")`**：每一筆訓練資料都由「沒看過它的那個 fold 的模型」給出機率 → 可以在訓練集上公平地選門檻。
- **PR 曲線**（`precision_recall_curve`）：門檻從高到低，每個門檻一組 (precision, recall)；回傳 `precision`、`recall`（長度 = 門檻數 + 1，最後一點是 (1, 0)）與 `thresholds`。

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_predict, learning_curve, validation_curve

alphas = np.logspace(-4, 2, 13)
tr_s, va_s = validation_curve(nb_pipe, X_train, y_train, param_name="nb__alpha", param_range=alphas, cv=cv, scoring="f1")
fig, ax = plt.subplots(figsize=(7, 4))
for name, s in [("train", tr_s), ("validation", va_s)]:
    ax.plot(alphas, s.mean(1), "o-", label=name)
    ax.fill_between(alphas, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
ax.set_xscale("log")
ax.set_xlabel("alpha (larger = simpler)")
ax.set_ylabel("F1 (spam)")
ax.set_title("Validation curve: alpha")
ax.legend()
plt.show()
display(pd.DataFrame({"train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}, index=alphas).round(4).T)

In [ ]:
# 搜尋：alpha × binary × ngram_range × min_df
param_grid = {
    "nb__alpha": [0.01, 0.03, 0.1, 0.3, 1.0],
    "vec__binary": [False, True],
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df": [1, 2],
}
gs = GridSearchCV(nb_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1)
gs.fit(X_train, y_train)
grid_res = pd.DataFrame(
    {
        "alpha": gs.cv_results_["param_nb__alpha"].astype(float),
        "binary": gs.cv_results_["param_vec__binary"].astype(bool),
        "ngram": [str(v) for v in gs.cv_results_["param_vec__ngram_range"]],
        "min_df": gs.cv_results_["param_vec__min_df"].astype(int),
        "mean": gs.cv_results_["mean_test_score"],
        "std": gs.cv_results_["std_test_score"],
    }
)
print("best_params_：", gs.best_params_, "  CV F1：", round(gs.best_score_, 4),
      " 詞彙表大小：", len(gs.best_estimator_["vec"].vocabulary_))
heat = grid_res.pivot_table(index="alpha", columns=["ngram", "binary", "min_df"], values="mean")
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("CV F1 (columns: ngram, binary, min_df)")
plt.show()
display(grid_res.sort_values("mean", ascending=False).head(8).round(4))

In [ ]:
# TODO 15（問答）validation curve 與搜尋
#
# (1) α 的 validation curve：
#     (a) train F1 在 α 很小時是多少？α 變大時 train、val 怎麼變？哪一邊過擬合、哪一邊欠擬合？跟 TODO 11 (2)(c) 的預測一樣嗎？
#     (b) val 最高在哪個 α？比預設的 α = 1 好多少（跟 std 比）？
# 答：
#
# (2) 搜尋結果：
#     (a) 最佳組合是什麼？ngram (1, 2) 讓詞彙表變大很多，F1 有變好嗎？
#     (b) 加入相鄰兩詞（"free entry"）會讓獨立假設更嚴重還是更輕微？（提示："free"、"entry"、"free entry" 三個特徵同時出現）
#         這對機率的校準可能有什麼影響？（TODO 12 (2)）
#     (c) 前八名之間的差距跟 std 比起來大嗎？如果用 1-SE，你會怎麼定義這裡的「簡單」？
# 答：

#### 最小例子：TODO 16（在 precision 限制下選門檻）

8 筆 CV 預測機率與真實標籤；要求 precision ≥ 0.75，在這個限制下 recall 最高的門檻。

1. **正確答案**：每個候選門檻都算一次 precision、recall，逐一檢查。目標：門檻 0.6、precision 0.75、recall 0.75（注意 precision 不是隨門檻單調上升的）。
2. **積木**：`precision_recall_curve` 回傳的三個陣列，長度差 1；去掉最後一點才能和 `thresholds` 對齊；布林遮罩。
3. **留給你**：在符合條件的門檻裡挑 recall 最大的。

In [ ]:
# 最小例子：TODO 16（在 precision 限制下選門檻）
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

y_mini = np.array([0, 0, 1, 0, 1, 0, 1, 1])
p_mini = np.array([0.05, 0.2, 0.4, 0.45, 0.6, 0.7, 0.8, 0.95])
target_mini = 0.75

# ---- 1. 正確答案：逐一檢查 ----
best_loop = None
for t in sorted(set(p_mini)):
    pred = (p_mini >= t).astype(int)
    pr, rc = precision_score(y_mini, pred, zero_division=0), recall_score(y_mini, pred)
    print(f"門檻 {t:<5} precision {pr:.3f}  recall {rc:.3f}  {'✓' if pr >= target_mini else ''}")
    if pr >= target_mini and (best_loop is None or rc > best_loop[2]):
        best_loop = (t, pr, rc)
print("正確答案：門檻、precision、recall =", best_loop)

# ---- 2. 積木 ----
prec_mini, rec_mini, thr_mini = precision_recall_curve(y_mini, p_mini)
print("\nprecision", prec_mini.round(3), len(prec_mini))
print("recall   ", rec_mini.round(3), len(rec_mini))
print("thresholds", thr_mini, len(thr_mini), "← 少一個：precision[-1] = 1、recall[-1] = 0 是額外加的終點")
ok_mini = prec_mini[:-1] >= target_mini
print("ok = precision[:-1] >= target", ok_mini)

# ---- 3. 留給你 ----
# 問題：在 ok 的門檻裡，用 np.argmax(rec[:-1][ok]) 拿到位置之後，對應的門檻是 thr[ok][...]？

In [ ]:
# 用 CV 的預測機率選門檻：在 precision ≥ 0.98 的限制下，recall 最高
TARGET_PRECISION = 0.98
best_nb = gs.best_estimator_
oof_proba = cross_val_predict(best_nb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_proba)

# TODO 16（程式）：選門檻
#   - ok = prec[:-1] >= TARGET_PRECISION（prec、rec 比 thr 多一個終點，要去掉才能對齊）
#   - 在 ok 裡挑 rec 最大的那個門檻 → threshold；沒有任何 ok 就用 0.5
raise NotImplementedError("TODO 16：選門檻")

oof_pred_05 = (oof_proba >= 0.5).astype(int)
oof_pred_t = (oof_proba >= threshold).astype(int)
print(f"選到的門檻：{threshold:.4g}")
print(f"門檻 0.5  ：precision {precision_score(y_train, oof_pred_05):.4f}  recall {recall_score(y_train, oof_pred_05):.4f}")
print(f"選到的門檻：precision {precision_score(y_train, oof_pred_t):.4f}  recall {recall_score(y_train, oof_pred_t):.4f}")
print("✅ 選到的門檻在 CV 上的 precision 應 ≥", TARGET_PRECISION, "：", precision_score(y_train, oof_pred_t) >= TARGET_PRECISION)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(rec, prec)
axes[0].axhline(TARGET_PRECISION, color="r", ls=":")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precision")
axes[0].set_title("PR curve (out-of-fold, train)")
axes[1].hist([oof_proba[ytr == 0], oof_proba[ytr == 1]], bins=np.linspace(0, 1, 26), label=["ham", "spam"], log=True)
axes[1].axvline(threshold, color="r", ls=":", label=f"threshold {threshold:.3g}")
axes[1].axvline(0.5, color="gray", ls="--", label="0.5")
axes[1].set_xlabel("out-of-fold P(spam)")
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
# 學習曲線：再多一點資料會更好嗎？
sizes_lc, tr_s, va_s = learning_curve(best_nb, X_train, y_train, cv=cv, scoring="f1", train_sizes=np.linspace(0.1, 1.0, 8))
fig, ax = plt.subplots(figsize=(7, 4))
for name, s in [("train", tr_s), ("validation", va_s)]:
    ax.plot(sizes_lc, s.mean(1), "o-", label=name)
    ax.fill_between(sizes_lc, s.mean(1) - s.std(1), s.mean(1) + s.std(1), alpha=0.2)
ax.set_xlabel("training samples")
ax.set_ylabel("F1 (spam)")
ax.set_title("Learning curve (tuned NB)")
ax.legend()
plt.show()
display(pd.DataFrame({"n": sizes_lc, "train": tr_s.mean(1), "val": va_s.mean(1), "val_std": va_s.std(1)}).round(4).T)

In [ ]:
# TODO 17（問答）門檻與學習曲線
#
# (1) 門檻：
#     (a) 選到的門檻是多少？比 0.5 高還是低？precision、recall 各怎麼變？
#     (b) 看右邊的直方圖：大部分訊息的 P(spam) 落在哪裡？為什麼 NB 的機率這麼「兩極」？（TODO 12 (2)）
#         這讓「調門檻」比較有效還是比較沒效？
#     (c) 為什麼門檻要用 cross_val_predict 的機率來選，而不是用測試集、也不是用 fit 在整個訓練集上的模型自己預測訓練集？
#     (d) 換一個情境：簡訊過濾器把 spam 丟進「垃圾匣」，使用者還看得到；或者直接刪除、使用者看不到。
#         這兩種情境的 TARGET_PRECISION 應該一樣嗎？
# 答：
#
# (2) 學習曲線：train 與 val 最後差多少？val 還在上升嗎？跟實驗 4 的結論一致嗎？
# 答：

---
## 6. 測試集評估

模型都已經用 CV 決定好，測試集只評估一次。另外放兩個前幾章的模型當對照（都沒有調參，只是參考）：

- **邏輯回歸**（第三章）：同樣的詞頻，$C = 1$。
- **決策樹**（第五章）：同樣的詞頻，`max_depth=10`。

#### 指標

- **precision、recall、F1**：以 spam = 1 為正類。
- **ROC AUC**：`classification_metrics` 給了機率就會算。
- `log_result`／`compare`：成績寫進 `results/sms_spam.csv`，依 F1 排序。

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from learning_map.metrics import classification_metrics
from learning_map.results import compare, log_result

BENCH = "sms_spam"
logreg = Pipeline([("vec", CountVectorizer()), ("lr", LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
tree = Pipeline([("vec", CountVectorizer()), ("tree", DecisionTreeClassifier(max_depth=10, random_state=0))]).fit(X_train, y_train)
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
bp = {k.split("__")[-1]: str(v) for k, v in gs.best_params_.items()}
best_proba_te = best_nb.predict_proba(X_test)[:, 1]

# 名稱: (預測, spam 的機率或 None, 要記下的參數)
eval_models = {
    "Dummy prior": (dummy.predict(X_test), dummy.predict_proba(X_test)[:, 1], {}),
    "Keyword rule": (KeywordRule().fit(X_train, y_train).predict(X_test), None, {}),
    "Multinomial NB a=1 (scratch)": (mnb_scratch.predict(Xte), mnb_scratch.predict_proba(Xte)[:, 1], {"alpha": 1.0}),
    "Bernoulli NB a=1 (scratch)": (bnb_scratch.predict(Xte), bnb_scratch.predict_proba(Xte)[:, 1], {"alpha": 1.0}),
    "Gaussian NB 6 numeric (scratch)": (gnb_scratch.predict(Fte), gnb_scratch.predict_proba(Fte)[:, 1], {}),
    "Multinomial NB a=1 (sklearn)": (nb_pipe.predict(X_test), nb_pipe.predict_proba(X_test)[:, 1], {"alpha": 1.0}),
    "Multinomial NB tuned (sklearn)": (best_nb.predict(X_test), best_proba_te, bp),
    "Multinomial NB tuned + threshold (sklearn)": ((best_proba_te >= threshold).astype(int), best_proba_te, {**bp, "threshold": round(threshold, 4)}),
    "LogReg C=1 (sklearn, ref)": (logreg.predict(X_test), logreg.predict_proba(X_test)[:, 1], {"C": 1.0}),
    "Tree depth 10 (sklearn, ref)": (tree.predict(X_test), tree.predict_proba(X_test)[:, 1], {"max_depth": 10}),
}

test_scores = {}
for name, (pred, prob, params) in eval_models.items():
    m = classification_metrics(y_test, pred, prob)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)

display(pd.DataFrame(test_scores).T)
compare(BENCH, "f1")

---
## 7. 診斷

樸素貝氏的預測可以**逐詞解釋**（3A 的 `explain`）：每個詞貢獻 $x_j(\log\theta_{\text{spam},j} - \log\theta_{\text{ham},j})$，加總就是兩類的分數差。

#### 本節名詞

- **false positive（FP）**：真的是 ham、被判成 spam（誤殺）。
- **false negative（FN）**：真的是 spam、被判成 ham（漏抓）。
- **OOV**（out of vocabulary）：訓練時沒看過的詞，`CountVectorizer` 直接忽略。

In [ ]:
from learning_map.metrics import plot_confusion_matrix

final_pred = (best_proba_te >= threshold).astype(int)
plot_confusion_matrix(y_test, final_pred, labels=["ham", "spam"])
plt.title("Tuned NB + threshold on test")
plt.show()

diag = pd.DataFrame({"text": X_test, "true": yte, "pred": final_pred, "P(spam)": best_proba_te})
diag["n_words"] = X_test.str.split().str.len()
diag["OOV ratio"] = 1 - np.asarray(best_nb["vec"].transform(X_test).sum(axis=1)).ravel() / np.maximum(
    np.asarray(CountVectorizer(ngram_range=best_nb["vec"].ngram_range).fit(X_test).transform(X_test).sum(axis=1)).ravel(), 1
)
print("False positives（ham → spam）：")
display(diag[(diag["true"] == 0) & (diag["pred"] == 1)].sort_values("P(spam)", ascending=False).round(4))
print("False negatives（spam → ham）：")
display(diag[(diag["true"] == 1) & (diag["pred"] == 0)].sort_values("P(spam)").round(4))
print("答對 vs 答錯的平均：")
display(diag.assign(correct=diag["true"] == diag["pred"]).groupby("correct")[["n_words", "OOV ratio", "P(spam)"]].mean().round(3))

In [ ]:
# 挑一則漏抓的 spam，看每個詞的貢獻（用 α = 1、單詞的手寫模型，比較好讀）
fn = diag[(diag["true"] == 1) & (diag["pred"] == 0)]
if len(fn):
    txt = fn.sort_values("P(spam)").iloc[0]["text"]
    print("文字：", txt)
    display(explain(txt).round(3))

In [ ]:
# TODO 18（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) 混淆矩陣：FP、FN 各幾筆？跟你在第 5 節選門檻時的取捨一致嗎？
#     (b) 讀漏抓的 spam：它們有什麼共同點？（長度、有沒有電話號碼、用詞像不像日常對話、OOV 比例）
#         把每一筆歸到：① 寫得像正常簡訊（內容本身難分）；② 關鍵資訊在詞袋看不到的地方（數字被切掉、詞的順序、拼字變形）；
#         ③ 訓練資料沒看過的詞（OOV）。哪一類最多？
#     (c) 看 explain 的表：那則漏抓的 spam，哪些詞把它往 ham 推？是「很像 ham 的詞」，還是只是「常見詞」？
# 答：
#
# (2) 改善方向：(b) 的每一類，各要改「資料」、「特徵（前處理、斷詞、n-gram、數字的處理）」還是「模型」？
#     例如：CountVectorizer 預設的 token_pattern 會把 "£1000" 切成什麼？電話號碼呢？
# 答：
#
# (3) 跟前幾章比（看第 6 節的表）：
#     (a) 調好的 NB、邏輯回歸、決策樹的 F1 各是多少？1115 筆測試集裡 spam 約 150 則（1 則 ≈ 0.7% 的 recall），差幾則？
#     (b) 比較表（填空）：
#
#         |                        | 樸素貝氏 | 邏輯回歸 | 決策樹 |
#         | 生成式／判別式         |          |          |        |
#         | 訓練成本               |          |          |        |
#         | 小資料時               |          |          |        |
#         | 特徵很多（7,700 個詞） |          |          |        |
#         | 機率的品質（校準）     |          |          |        |
#         | 怎麼解釋一個預測       |          |          |        |
#
#     (c) 什麼情況下你會選樸素貝氏？什麼情況下不會？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

SMS Spam Collection（train / test 依載入結果；CV = 5-fold 分層；主要指標 = spam 類的 F1）。

| 方法 | 做法 | 關鍵性質 | CV F1 | test F1 |
| --- | --- | --- | --- | --- |
| Dummy prior | 輸出 spam 比例 | | | |
| 關鍵字規則 | 5 位以上數字或關鍵字 | | | |
| Multinomial NB $\alpha = 1$（手寫） | 數數 + Laplace 平滑 + log-sum-exp | | | |
| Bernoulli NB $\alpha = 1$（手寫） | 有沒有出現；稀疏技巧 | | | |
| Gaussian NB（手寫） | 6 個數值特徵 | | | |
| Multinomial NB 調參 + 門檻 | alpha × binary × n-gram × min_df；precision ≥ 0.98 | | | |
| 邏輯回歸（對照） | 詞頻，$C = 1$ | | — | |
| 決策樹（對照） | 詞頻，深度 10 | | — | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**機率與估計**
- ☐ 平滑的貝氏解釋：Dirichlet 先驗、MAP 估計；$\alpha$ 是「假想的計數」
- ☐ 多項式係數為什麼可以丟掉；文件長度在 Multinomial NB 裡的角色
- ☐ NB 的機率校準：`CalibratedClassifierCV`（Platt scaling、isotonic）
- ☐ 獨立假設什麼時候「錯了也沒關係」：Domingos & Pazzani (1997) 對 0-1 loss 的分析

**模型**
- ☐ Complement NB（Rennie et al. 2003）、Transformed Weight-normalized NB 的細節
- ☐ NB-SVM（Wang & Manning 2012）：用 NB 的 log ratio 當特徵權重再訓練判別式模型
- ☐ 生成式 vs 判別式的理論：Ng & Jordan (2001)；LDA／QDA（Gaussian 但不假設獨立）
- ☐ 半監督：用 EM 讓 NB 從沒標記的文字學習（Nigam et al. 2000）

**文字前處理**
- ☐ 斷詞（`token_pattern`）、數字與金額的正規化、stemming／lemmatization、停用詞
- ☐ 字元 n-gram（對拼字變形更穩）、hashing trick（`HashingVectorizer`）
- ☐ 線上學習：`partial_fit` 只更新計數

**評估**
- ☐ 重複資料對評估的影響（去重後再切分）、依時間切分
- ☐ 成本敏感的決策：把 FP、FN 的代價寫成數字，選期望成本最小的門檻